## Bronze Layer — Raw Data Ingestion

The **Bronze Layer** is the first stage of the Medallion Architecture. Data is ingested from source systems with **full fidelity** — no transformations, no business logic. Audit metadata columns (`_ingestion_timestamp`, `_ingested_by`, `_source_file`, `_source_filename`, `_source_system`) are added to every record for lineage and traceability.

| Property | Value |
|---|---|
| Source | Azure Data Lake Storage Gen2 — `raw` container |
| Catalog | `dev.bronze` |
| Table | `dev.bronze.sales` |
| Format | Delta (managed, stored at ADLS Gen2 managed location) |
| Load strategy | Full reload (`CREATE OR REPLACE TABLE`) |

In [0]:
%sql 
CREATE CATALOG IF NOT EXISTS dev managed location 'abfss://raw@devstgaccdeprj.dfs.core.windows.net/'

In [0]:
%sql CREATE SCHEMA IF NOT EXISTS dev.bronze;

CREATE OR REPLACE TABLE dev.bronze.sales
AS
SELECT
    *,
    current_timestamp()                                                    AS _ingestion_timestamp,
    current_user()                                                         AS _ingested_by,
    'abfss://raw@devstgaccdeprj.dfs.core.windows.net/Sales.parquet'       AS _source_file,
    'Sales.parquet'                                                        AS _source_filename,
    'ADLS Gen2'                                                            AS _source_system
FROM PARQUET.`abfss://raw@devstgaccdeprj.dfs.core.windows.net/Sales.parquet`;

In [0]:
df = spark.table("dev.bronze.sales")
row_count = df.count()
col_count  = len(df.columns)

print(f"Table     : dev.bronze.sales")
print(f"Row Count : {row_count:,}")
print(f"Columns   : {col_count}")
print()
df.printSchema()

In [0]:
from pyspark.sql.functions import col, count, when

df = spark.table("dev.bronze.sales")
total = df.count()

# Exclude audit metadata columns
data_cols = [c for c in df.columns if not c.startswith("_")]

results = []
for c in data_cols:
    null_cnt = df.filter(col(c).isNull() | (col(c) == "")).count()
    results.append({"column": c, "null_or_empty": null_cnt, "null_pct": round(null_cnt / total * 100, 2)})

results.sort(key=lambda x: x["null_or_empty"], reverse=True)

print(f"{'Column':<35} {'Null/Empty':>12} {'%':>8}")
print("-" * 58)
for r in results:
    flag = " ⚠️" if r["null_pct"] > 0 else ""
    print(f"{r['column']:<35} {r['null_or_empty']:>12,} {r['null_pct']:>7.2f}%{flag}")

In [0]:
%sql
SELECT
    COUNT(*)                                                  AS total_records,
    COUNT(DISTINCT Item_Identifier)                           AS unique_items,
    COUNT(DISTINCT Outlet_Identifier)                         AS unique_outlets,
    ROUND(AVG(CAST(Item_Outlet_Sales         AS DOUBLE)), 2)  AS avg_sales,
    ROUND(MIN(CAST(Item_Outlet_Sales         AS DOUBLE)), 2)  AS min_sales,
    ROUND(MAX(CAST(Item_Outlet_Sales         AS DOUBLE)), 2)  AS max_sales,
    MIN(CAST(Outlet_Establishment_Year AS INT))               AS oldest_outlet_year,
    MAX(CAST(Outlet_Establishment_Year AS INT))               AS newest_outlet_year,
    COUNT(CASE WHEN Item_Weight  IS NULL THEN 1 END)          AS null_item_weight,
    COUNT(CASE WHEN Outlet_Size  IS NULL THEN 1 END)          AS null_outlet_size,
    MIN(_ingestion_timestamp)                                 AS ingestion_timestamp
FROM dev.bronze.sales

In [0]:
df = spark.table("dev.bronze.sales")

categorical_cols = [
    "Item_Fat_Content",
    "Item_Type",
    "Outlet_Identifier",
    "Outlet_Size",
    "Outlet_Location_Type",
    "Outlet_Type",
]

print(f"{'Column':<30} {'#Distinct':>10}   Values")
print("-" * 80)
for c in categorical_cols:
    vals = sorted([r[c] for r in df.select(c).distinct().collect() if r[c] is not None])
    print(f"{c:<30} {len(vals):>10}   {vals}")